In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler   import OneCycleLR, ExponentialLR

import time
#from tqdm import tqdm
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.preprocessing import MinMaxScaler, StandardScaler

from model_rul.RulModel import ModelHI
from model_rul.RulExperiment    import ExperimentRul

c:\Users\CAEX\Desktop\Diego\Tesis\llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_hdf('dataset\processed_crusher.h5')
df['cycles'] =  df.groupby('mantenciones').cumcount() #trajectory

In [3]:
manipulated_cr1=[
                'corriente(A)',
                'nivel_tolva_compensacion(%)',
                'Motor_Fase1(A)',
                'Motor_Fase2(A)',
                'Motor_Fase3(A)',
                'Motor_Fase1(V)',
                'Motor_Fase2(V)',
                'Lubricación_principal(A)',
                'Lubricación_respaldo(A)', 
                ]


efficiency_cr1 = [
                'posicion_poste(%)',
                'Piñón_ext(°C)',
                'Piñón_int(°C)',
                'Lubricación_depósito(%)',
                'Lubricación_entrada_sup(psi)',
                'Lubricación_entrada_inf(psi)',
                'Lubricación_principal(psi)',
                'presion_estanque_aire(bar)'
                ]

new_efficiency_cr1 = ['Eficiencia_Rendimiento',    'Vibracion_piñon_int_norm',  'Vibracion_piñon_ext_norm', 
                      'Desviacion_Vibracion_Int',  'Desviacion_Vibracion_Ext',  'Discrepancia_temperatura_lubricacion']
        
new_manipulable_cr1 = ['Discrepancia_Corriente_Motor', 'Discrepancia_Voltaje_Motor', 'Sistema Lubricación(A)']

In [4]:
window_size = 48
input_columns = manipulated_cr1+new_manipulable_cr1
output_columns = efficiency_cr1+new_efficiency_cr1


model_params = {'out_channels': [128,64,12], 
                'dim_x': len(output_columns),
                'dim_w': len(input_columns), 
                'window_size': window_size,
                'z_size':1, 
                'kernel_size':85, 
                'stride':2, 
                'alpha': 1, 
                'constraint':'F'}

config = {'data': df, 
        'input_columns': input_columns,
        'output_columns': output_columns,
        'window_size': window_size,
        'batch_size': 64,
        'criterion': nn.L1Loss(),
        'epochs': 10,
        'lr': 1e-4,
        'plot_loss': False,
        'directory': 'saved_models/rul_model.pt',
        'model_params': model_params}

experiment = ExperimentRul(**config)
experiment.train()
experiment.extract_latent_space()

Número de parámetros a entrenar: 1691163


Epoch: 01: 100%|██████████| 435/435 [00:04<00:00, 98.28it/s] 


	Epoch Time: 0m 4s | Train Loss: 0.907 | Train MSE: 0.840 | Train MAE: 0.695


Epoch: 02: 100%|██████████| 435/435 [00:02<00:00, 186.05it/s]


	Epoch Time: 0m 2s | Train Loss: 0.751 | Train MSE: 0.702 | Train MAE: 0.613


Epoch: 03: 100%|██████████| 435/435 [00:02<00:00, 185.25it/s]


... (11 outputs left out)

Epoch: 09: 100%|██████████| 435/435 [00:02<00:00, 185.77it/s]


	Epoch Time: 0m 2s | Train Loss: 0.366 | Train MSE: 0.320 | Train MAE: 0.358


Epoch: 10: 100%|██████████| 435/435 [00:02<00:00, 182.61it/s]


	Epoch Time: 0m 2s | Train Loss: 0.358 | Train MSE: 0.314 | Train MAE: 0.352


In [5]:
df_sliding = experiment.rul_processing.df_sliding
df_sliding['rul'] = experiment.z
df_ = df.join(df_sliding[['tiempo', 'rul']].set_index('tiempo'), on='tiempo')
df_['rul_avg'] = df_['rul'].rolling(window=6, min_periods=1).mean()
df_['index'] = df_.index.values

fig = px.line(df_, x='index', y='rul', color='trajectory')
fig.update_layout(
            template='plotly_white',
            plot_bgcolor='rgba(255, 255, 255, 1)',
            paper_bgcolor='rgba(255, 255, 255, 1)',
            xaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor='black'),
            yaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor='black'),
            )
fig.show()

In [6]:
def weibull_distribution(t, alpha=1):
    A = 3.24689126e-04
    B = 6.75139693e-01
    C = 1#.02044605e+00
    BETA = 3.5
    P = 0.5
    
    # Calcula HI
    HI = C - ((t * (-np.log(1 - P))**(-1/BETA)) * A)**B
    
    # Aplica la máscara para asegurar que HI no sea menor que 0
    HI = np.where(HI < 0, 0.0, HI)
    
    return HI


In [7]:
df_['g(t)'] = df_['cycles'].apply(weibull_distribution)
fig = px.line(df_, x='index', y='g(t)', color='trajectory')
fig.update_layout(
            template='plotly_white',
            plot_bgcolor='rgba(255, 255, 255, 1)',
            paper_bgcolor='rgba(255, 255, 255, 1)',
            xaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor='black'),
            yaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor='black'),
            )
fig.show()